# LyricInsight 감정 분석 모델 학습 (Colab용)

이 노트북은 LyricInsight 프로젝트의 한국어 가사 감정 분석 모델(`klue/roberta-base`)을 학습하기 위해 작성되었습니다.

## 1. 사전 준비
이 노트북을 실행하기 전에 왼쪽 **파일(Files)** 탭에 다음 파일들을 업로드해주세요:
1. `train.jsonl`
2. `val.jsonl`
3. `labels.json`

위 파일들은 로컬 프로젝트의 `d:\LyricInsight\data\processed_kpoem\` 경로에 있습니다.

In [ ]:
# 2. 필요 라이브러리 설치
!pip install transformers datasets scikit-learn accelerate torch

In [ ]:
import json
from pathlib import Path
import numpy as np
from sklearn.metrics import f1_score
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
import torch

# GPU 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# 3. 데이터 로드 및 설정

# Colab 환경에서는 현재 디렉토리(/content/)를 사용
DATA_DIR = Path(".")

# 데이터 파일 경로
TRAIN_PATH = DATA_DIR / "train.jsonl"
VAL_PATH = DATA_DIR / "val.jsonl"
LABELS_PATH = DATA_DIR / "labels.json"

# 출력 모델 경로
OUT_MODEL_DIR = Path("./emotion_v2")

def load_jsonl(path: Path):
    rows = []
    if not path.exists():
        raise FileNotFoundError(f"{path} 파일을 찾을 수 없습니다. Colab의 파일 탭에 업로드했는지 확인해주세요.")

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

if not LABELS_PATH.exists():
    print("Warning: labels.json not found. Please upload it.")
else:
    label_names = json.loads(LABELS_PATH.read_text(encoding="utf-8"))
    print("Labels:", label_names)
    print("Num labels:", len(label_names))

In [ ]:
# 4. 데이터셋 준비

MODEL_NAME = "klue/roberta-base"

train_rows = load_jsonl(TRAIN_PATH)
val_rows = load_jsonl(VAL_PATH)

train_ds = Dataset.from_list(train_rows)
val_ds = Dataset.from_list(val_rows)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

def cast_labels(batch):
    batch["labels"] = np.array(batch["labels"], dtype=np.float32)
    return batch

train_ds = train_ds.map(cast_labels)
val_ds = val_ds.map(cast_labels)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Map:   0%|          | 0/5604 [00:00<?, ? examples/s]

Map:   0%|          | 0/1401 [00:00<?, ? examples/s]

Map:   0%|          | 0/5604 [00:00<?, ? examples/s]

Map:   0%|          | 0/1401 [00:00<?, ? examples/s]

In [ ]:
# 5. 모델 학습 설정 및 시작

# Fix for NameError: name 'label_names' is not defined
# Ensure label_names is loaded since labels.json is now available.
label_names = json.loads(LABELS_PATH.read_text(encoding="utf-8"))

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = sigmoid(logits)

    # threshold
    thr = 0.3
    preds = (probs >= thr).astype(int)

    micro = f1_score(labels, preds, average="micro", zero_division=0)
    macro = f1_score(labels, preds, average="macro", zero_division=0)

    return {
        "f1_micro": micro,
        "f1_macro": macro,
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    problem_type="multi_label_classification",
)

args = TrainingArguments(
    output_dir=str(OUT_MODEL_DIR),
    eval_strategy="epoch", # Changed from evaluation_strategy
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16, # Colab GPU라 배치 사이즈 좀 늘림
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    save_total_limit=2,
    fp16=True, # GPU 가속 활용
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at klue/roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: ERROR Invalid API key: API key must have 40+ characters, has 8.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.291300,0.284322,0.287977,0.061996
2,0.256100,0.253093,0.446970,0.165972
3,0.236300,0.235473,0.504536,0.257193
4,0.212500,0.230601,0.514072,0.275328
5,0.208600,0.227253,0.522148,0.294787


TrainOutput(global_step=1755, training_loss=0.2519869087768076, metrics={'train_runtime': 344.2482, 'train_samples_per_second': 81.395, 'train_steps_per_second': 5.098, 'total_flos': 1843787975086080.0, 'train_loss': 0.2519869087768076, 'epoch': 5.0})

In [ ]:
# 6. 모델 저장

OUT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUT_MODEL_DIR))
tokenizer.save_pretrained(str(OUT_MODEL_DIR))

# labels.json 복사
(OUT_MODEL_DIR / "labels.json").write_text(
    json.dumps(label_names, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Model saved to", OUT_MODEL_DIR)

Model saved to emotion_v2


In [ ]:
# 7. 모델 다운로드 압축
!zip -r emotion_v2.zip emotion_v2

  adding: emotion_v2/ (stored 0%)
  adding: emotion_v2/tokenizer_config.json (deflated 75%)
  adding: emotion_v2/checkpoint-1404/ (stored 0%)
  adding: emotion_v2/checkpoint-1404/model.safetensors (deflated 11%)
  adding: emotion_v2/checkpoint-1404/config.json (deflated 68%)
  adding: emotion_v2/checkpoint-1404/training_args.bin (deflated 53%)
  adding: emotion_v2/checkpoint-1404/rng_state.pth (deflated 26%)
  adding: emotion_v2/checkpoint-1404/optimizer.pt (deflated 25%)
  adding: emotion_v2/checkpoint-1404/trainer_state.json (deflated 75%)
  adding: emotion_v2/checkpoint-1404/scaler.pt (deflated 64%)
  adding: emotion_v2/checkpoint-1404/scheduler.pt (deflated 61%)
  adding: emotion_v2/model.safetensors (deflated 11%)
  adding: emotion_v2/special_tokens_map.json (deflated 85%)
  adding: emotion_v2/labels.json (deflated 47%)
  adding: emotion_v2/config.json (deflated 68%)
  adding: emotion_v2/runs/ (stored 0%)
  adding: emotion_v2/runs/Feb02_07-24-18_d5b97f1049a9/ (stored 0%)
  adding:

In [ ]:
# 8. 다운로드 (브라우저 다운로드 창이 뜨지 않으면 왼쪽 파일 탭에서 직접 다운로드 가능)
from google.colab import files
files.download('emotion_v2.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>